# Calculate the FK submesoscale heat flux and save
Ce = 0.1485399
LfMin = 1000
Lmax = 110000
inv_ tau = 2.0e-6

## Save the kpphbl, T, S, Eta for calculate the FK Heat flux

In [7]:
nStep1, nStep2, dS = 1200060 + 180, 1279080, 180
time_index = np.arange(nStep1, nStep2, dS, dtype=int)
print(time_index[-1])
print(len(time_index))

1278900
438


In [32]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

import xarray as xr
import numpy as np

from src.utils import make_grid
from src.utils import build_coarse_grid
from src.utils import read_mitgcm
from src.utils import coarsen_UVWTS
from MITgcmutils import mds
from src.utils import mld_avg


# ----------------------------
# path
# ----------------------------
path_scratch = "/scratch/jczhang/"
path_MITgcm = "MITgcm_1km/verification/tutorial_reentrant_channel/output"
grid_dir = path_scratch + path_MITgcm

# ----------------------------
# load grid
# ----------------------------
box_i = 50
box_j = 50

ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)
ds_coarse = build_coarse_grid.build_coarse_grid_staggered(
    ds_fine,
    box_j=box_j,
    box_i=box_i,
)
# time index
nStep1, nStep2, dS = 1202220, 1279800 + 1, 180
time_index = np.arange(nStep1, nStep2, dS, dtype=int)

def load_12snap_avg_coarse_U(
    grid_dir,
    ds_fine,
    time_index,
    block_size=6,
    box_j=50,
    box_i=50,
    out_npz="U_12snap_avg_coarse.npz",
    allow_incomplete_last_block=False,
):
    """
    Load U snapshots, average every 12 snapshots, coarse-grain the averaged U,
    and save as npz.

    Output npz contains:
        U_coarse: shape (time_avg, Z, Y, X)
        block_start_time_index: initial MITgcm time index of each 12-snapshot block
        block_end_time_index: final MITgcm time index of each 12-snapshot block
        block_size: number of snapshots averaged per block
    """

    time_index = np.asarray(time_index).astype(int)

    n_total = len(time_index)

    if allow_incomplete_last_block:
        n_blocks = int(np.ceil(n_total / block_size))
    else:
        n_blocks = n_total // block_size

    if n_blocks == 0:
        raise ValueError(
            f"Not enough snapshots. Got {n_total}, but block_size={block_size}."
        )

    U_coarse_list = []
    block_start_time_index = []
    block_end_time_index = []

    for iblock in range(n_blocks):
        start = iblock * block_size
        end = min((iblock + 1) * block_size, n_total)

        if (end - start) < block_size and not allow_incomplete_last_block:
            continue

        times_block = time_index[start:end]

        print(
            f"Block {iblock + 1}/{n_blocks}: "
            f"averaging {len(times_block)} snapshots, "
            f"start={times_block[0]}, end={times_block[-1]}"
        )

        # ----------------------------
        # load and accumulate 12 snapshots
        # ----------------------------
        U_sum = None

        U_list = []
        
        for t in times_block:
            U = read_mitgcm._ocean_var(
                grid_dir,
                "Eta",
                int(t),
                ("j", "i"),
            ).astype("float64")
            U_list.append(U)
        
        U_avg = xr.concat(U_list, dim="time_block").mean(dim="time_block")

        # ----------------------------
        # coarse averaged U
        # ----------------------------
        U_coarse = coarsen_UVWTS.coarse_field(
            "Eta",
            U_avg,
            ds_fine,
            box_j,
            box_i,
        )

        # convert to numpy
        # expected dims: (k, j, i_g) or similar
        U_coarse_np = np.asarray(U_coarse.values, dtype=np.float32)

        U_coarse_list.append(U_coarse_np)
        block_start_time_index.append(int(times_block[0]))
        block_end_time_index.append(int(times_block[-1]))

    # ----------------------------
    # stack to shape (time_avg, Z, Y, X)
    # ----------------------------
    U_coarse_all = np.stack(U_coarse_list, axis=0).astype(np.float32)

    block_start_time_index = np.asarray(block_start_time_index, dtype=np.int64)
    block_end_time_index = np.asarray(block_end_time_index, dtype=np.int64)

    print("Final U_coarse shape:", U_coarse_all.shape)
    print("block_start_time_index shape:", block_start_time_index.shape)

    # ----------------------------
    # save
    # ----------------------------
    np.savez_compressed(
        out_npz,
        U_coarse=U_coarse_all,
        block_start_time_index=block_start_time_index,
        block_end_time_index=block_end_time_index,
        block_size=np.asarray(block_size, dtype=np.int64),
    )

    print(f"Saved to: {out_npz}")

    return U_coarse_all, block_start_time_index, block_end_time_index

In [33]:
U_coarse_all, block_start_time_index, block_end_time_index = load_12snap_avg_coarse_U(
    grid_dir=grid_dir,
    ds_fine=ds_fine,
    time_index=time_index,
    block_size=6,
    box_j=50,
    box_i=50,
    out_npz="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/Eta_12snap_avg_coarse.npz",
    allow_incomplete_last_block=False,
)

Block 1/72: averaging 6 snapshots, start=1202220, end=1203120
Block 2/72: averaging 6 snapshots, start=1203300, end=1204200
Block 3/72: averaging 6 snapshots, start=1204380, end=1205280
Block 4/72: averaging 6 snapshots, start=1205460, end=1206360
Block 5/72: averaging 6 snapshots, start=1206540, end=1207440
Block 6/72: averaging 6 snapshots, start=1207620, end=1208520
Block 7/72: averaging 6 snapshots, start=1208700, end=1209600
Block 8/72: averaging 6 snapshots, start=1209780, end=1210680
Block 9/72: averaging 6 snapshots, start=1210860, end=1211760
Block 10/72: averaging 6 snapshots, start=1211940, end=1212840
Block 11/72: averaging 6 snapshots, start=1213020, end=1213920
Block 12/72: averaging 6 snapshots, start=1214100, end=1215000
Block 13/72: averaging 6 snapshots, start=1215180, end=1216080
Block 14/72: averaging 6 snapshots, start=1216260, end=1217160
Block 15/72: averaging 6 snapshots, start=1217340, end=1218240
Block 16/72: averaging 6 snapshots, start=1218420, end=1219320
B

## Save kpphbl - no need to average

In [22]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

import xarray as xr
import numpy as np

from src.utils import make_grid
from src.utils import build_coarse_grid
from src.utils import read_mitgcm
from src.utils import coarsen_UVWTS
from MITgcmutils import mds
from src.utils import mld_avg


# ----------------------------
# path
# ----------------------------
path_scratch = "/scratch/jczhang/"
path_MITgcm = "MITgcm_1km/verification/tutorial_reentrant_channel/output"
grid_dir = path_scratch + path_MITgcm

# ----------------------------
# load grid
# ----------------------------
box_i = 50
box_j = 50

ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)
ds_coarse = build_coarse_grid.build_coarse_grid_staggered(
    ds_fine,
    box_j=box_j,
    box_i=box_i,
)

# ----------------------------
# time index
# ----------------------------
nStep1, nStep2, dS = 1203120, 1279800 + 1, 1080
time_index = np.arange(nStep1, nStep2, dS, dtype=int)


def load_coarse_kpphbl(
    grid_dir,
    ds_fine,
    time_index,
    box_j=50,
    box_i=50,
    out_npz="kpphbl_coarse.npz",
):
    """
    Load kpphbl snapshots (already 12h output), coarse-grain each snapshot,
    and save as npz.

    Output npz contains:
        kpphbl_coarse: shape (time, Y, X)
        time_index: original MITgcm time index for each snapshot
    """

    time_index = np.asarray(time_index).astype(int)

    kpphbl_coarse_list = []
    saved_time_index = []

    diag_path = os.path.join(path_scratch, path_MITgcm, "Diags", "DiagMXL_2d")

    for it, t in enumerate(time_index):
        print(f"Loading {it+1}/{len(time_index)}: t = {t}")

        # ----------------------------
        # read one 12h kpphbl snapshot
        # ----------------------------
        diagMXL = mds.rdmds(diag_path, int(t))

        # assuming diagMXL[1,:,:] is kpphbl
        kpphbl = xr.DataArray(
            diagMXL[1, :, :],
            dims=("j", "i"),
            name="kpphbl",
        ).astype("float64")

        # ----------------------------
        # coarse
        # treat it like a tracer/center field
        # ----------------------------
        kpphbl_coarse = coarsen_UVWTS.coarse_field(
            "kpphbl",
            kpphbl,
            ds_fine,
            box_j,
            box_i,
        )

        kpphbl_coarse_np = np.asarray(kpphbl_coarse.values, dtype=np.float32)

        kpphbl_coarse_list.append(kpphbl_coarse_np)
        saved_time_index.append(int(t))

    # ----------------------------
    # stack to shape (time, Y, X)
    # ----------------------------
    kpphbl_coarse_all = np.stack(kpphbl_coarse_list, axis=0).astype(np.float32)
    saved_time_index = np.asarray(saved_time_index, dtype=np.int64)

    print("Final kpphbl_coarse shape:", kpphbl_coarse_all.shape)
    print("time_index shape:", saved_time_index.shape)

    # ----------------------------
    # save
    # ----------------------------
    np.savez_compressed(
        out_npz,
        kpphbl_coarse=kpphbl_coarse_all,
        time_index=saved_time_index,
    )

    print(f"Saved to: {out_npz}")

    return kpphbl_coarse_all, saved_time_index

In [23]:
kpphbl_coarse_all, saved_time_index = load_coarse_kpphbl(
    grid_dir=grid_dir,
    ds_fine=ds_fine,
    time_index=time_index,
    box_j=50,
    box_i=50,
    out_npz="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/kpphbl_12snap_avg_coarse.npz",
)

Loading 1/72: t = 1203120
Loading 2/72: t = 1204200
Loading 3/72: t = 1205280
Loading 4/72: t = 1206360
Loading 5/72: t = 1207440
Loading 6/72: t = 1208520
Loading 7/72: t = 1209600
Loading 8/72: t = 1210680
Loading 9/72: t = 1211760
Loading 10/72: t = 1212840
Loading 11/72: t = 1213920
Loading 12/72: t = 1215000
Loading 13/72: t = 1216080
Loading 14/72: t = 1217160
Loading 15/72: t = 1218240
Loading 16/72: t = 1219320
Loading 17/72: t = 1220400
Loading 18/72: t = 1221480
Loading 19/72: t = 1222560
Loading 20/72: t = 1223640
Loading 21/72: t = 1224720
Loading 22/72: t = 1225800
Loading 23/72: t = 1226880
Loading 24/72: t = 1227960
Loading 25/72: t = 1229040
Loading 26/72: t = 1230120
Loading 27/72: t = 1231200
Loading 28/72: t = 1232280
Loading 29/72: t = 1233360
Loading 30/72: t = 1234440
Loading 31/72: t = 1235520
Loading 32/72: t = 1236600
Loading 33/72: t = 1237680
Loading 34/72: t = 1238760
Loading 35/72: t = 1239840
Loading 36/72: t = 1240920
Loading 37/72: t = 1242000
Loading 38

## FK functions

In [43]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("/scratch/jczhang/submeso_data_driven").resolve()))

from src.utils import make_grid
from src.utils import build_coarse_grid 
from src.utils import read_mitgcm 
import xarray as xr 
import numpy as np 
import gsw 
import importlib 
from src.utils import coarsen_UVWTS 
from MITgcmutils import mds 
importlib.reload(coarsen_UVWTS) # force reload

def buo_coarse(
    ds_grid: xr.Dataset,
    ds_ocean: xr.Dataset,
    boundary_Y: str = "fill"
) -> xr.DataArray:

    # merge the two dataset
    ds = xr.merge([ds_grid, ds_ocean])
    grid = make_grid.make_grid_periodic_x(ds)
    # water collumn
    # Etabar: sea-surface height (j,i) [m]; broadcast add → z_c(k,j,i)
    z_rel = ds["RC"] - ds["Eta"]  # RC is negative, RC - Eta gives the depth of water column
    
    # TEOS10 EOS
    # model output is salinity and potential temperature
    # gsw function need absolute salinity and conservative temperature
    lat0, lon0 = -55.0, 153.0
    lat2d = xr.full_like(ds["T"].isel(k=0), lat0)               # (j,i)
    lon2d = xr.full_like(ds["T"].isel(k=0), np.mod(lon0, 360.)) # (j,i)

    # TEOS-10 sea pressure from depth (returns dbar)
    p_dbar = xr.apply_ufunc(
        gsw.p_from_z, z_rel, lat2d,
        dask="parallelized",
        output_dtypes=[ds["T"].dtype],
    )


    # --- convert SP, θ -> SA, CT -> ρ ---
    SA  = xr.apply_ufunc(gsw.SA_from_SP, ds["S"], p_dbar, lon2d, lat2d,
                         dask='parallelized', output_dtypes=[ds["S"].dtype])
    CT  = xr.apply_ufunc(gsw.CT_from_pt, SA, ds["T"],
                         dask='parallelized', output_dtypes=[ds["T"].dtype])
    rho = xr.apply_ufunc(gsw.rho, SA, CT, p_dbar,
                         dask='parallelized', output_dtypes=[ds["T"].dtype])  # kg m^-3
    # constants
    g = 9.81 # gravity
    rho0 = 1035 # density
    # buoyancy
    b = (-g * (rho - rho0) / rho0).where(ds["hFacC"] > 0)  # m s^-2
    b.name = "buoyancy"
    b.attrs.update(units="m s^-2", long_name="buoyancy (TEOS-10)")
    
    # ---- buoyancy gradient ----
    # Face wet masks (avoid bleeding across land/blocked faces)
    wetU = ds["hFacW"] > 0
    wetV = ds["hFacS"] > 0
    dbdx_u = (grid.diff(b, "X") / ds["DXC"]).where(wetU)                   # (k,j,i_g)
    dbdy_v = (grid.diff(b, "Y", boundary=boundary_Y) / ds["DYC"]).where(wetV)  # (k,j_g,i)
    return b, dbdx_u, dbdy_v

In [44]:
def _k_name(da): # Find the vertical dim name that matches your data 
    for cand in ("k", "Z", "z", "lev", "k_c"): 
        if cand in da.dims: 
            return cand 
            # fallback: assume first dim is vertical (adjust if needed) 
    return da.dims[0]

def mld_integrals_for_MEI( dbdx_u: xr.DataArray, # (k,j,i_g) ∂b/∂x at U 
                           dbdy_v: xr.DataArray, # (k,j_g,i) ∂b/∂y at V 
                           kpphbl_C: xr.DataArray, # (j,i) ML depth at C 
                           ds_grid: xr.Dataset, 
                           grid, 
                           sigma_is_buoyancy: bool = True, 
                           g: float = 9.81, 
                           rho0: float = 1035.0, 
                         ): 
    # ---- 0) Standardize the vertical index 'k' everywhere ---- 
    RF = ds_grid["RF"] # (k_p1), negative downward , 50
    K = RF.sizes["k_p1"] - 1  # 49
    k_index = xr.DataArray(np.arange(K), dims=("k",), name="k") 
    ku = _k_name(dbdx_u) 
    kv = _k_name(dbdy_v) 
    dbdx_u = (dbdx_u.rename({ku: "k"}) .assign_coords(k=k_index) .reset_coords(drop=True)) 
    dbdy_v = (dbdy_v.rename({kv: "k"}) .assign_coords(k=k_index) .reset_coords(drop=True)) 
    # Mixed-layer depth at faces 
    H_C = kpphbl_C.clip(min=0.0) 
    H_U = grid.interp(H_C, "X", to="left") # (j,i_g) 
    H_V = grid.interp(H_C, "Y", to="left") # (j_g,i) 
    # ---- 1) Interfaces as positive-down depths with SAME 'k' coords ---- 
    ztop1d = (-RF.isel(k_p1=slice(0, -1))).rename(k_p1="k").assign_coords(k=k_index) # 0 to Nr -1
    zbot1d = (-RF.isel(k_p1=slice(1, None))).rename(k_p1="k").assign_coords(k=k_index) # 1 to Nr
    # Broadcast to faces (will use positional matching now that 'k' is unified) 
    ztop_U = ztop1d.broadcast_like(dbdx_u) # (k,j,i_g) 
    zbot_U = zbot1d.broadcast_like(dbdx_u) 
    ztop_V = ztop1d.broadcast_like(dbdy_v) # (k,j_g,i) 
    zbot_V = zbot1d.broadcast_like(dbdy_v) 
    # Partial-cell thickness on faces 
    thkU = (zbot_U - ztop_U) * ds_grid["hFacW"] 
    thkV = (zbot_V - ztop_V) * ds_grid["hFacS"] 
    # ---- 2) Overlap thickness with [0, H_face] (no xr.minimum) ---- 
    H3U = H_U.broadcast_like(dbdx_u) 
    H3V = H_V.broadcast_like(dbdy_v) 
    # min(H, zbot) = where(H <= zbot, H, zbot) 
    minHU = xr.where(H3U <= zbot_U, H3U, zbot_U) 
    minHV = xr.where(H3V <= zbot_V, H3V, zbot_V) 
    wU = xr.where(H3U > ztop_U, minHU - ztop_U, 0.0) 
    wV = xr.where(H3V > ztop_V, minHV - ztop_V, 0.0) 
    # clip and enforce partial thickness 
    wU = xr.where(wU > 0, xr.where(wU < thkU, wU, thkU), 0.0) 
    wV = xr.where(wV > 0, xr.where(wV < thkV, wV, thkV), 0.0) 
    # ---- 3) Integrals over 0..H (∑ sigma * dzLoc) ---- 
    intX = (dbdx_u.fillna(0.0) * wU).sum("k") # (j,i_g) 
    intY = (dbdy_v.fillna(0.0) * wV).sum("k") # (j_g,i) 
    # ---- 4) Convert if sigma are density-gradients (yours are buoyancy-gradients) ---- 
    if sigma_is_buoyancy: 
        dBuoyX_Hu, dBuoyY_Hv = intX, intY 
        out_units = "s^-2 m" 
    else: 
        fac = -g / rho0 
        dBuoyX_Hu, dBuoyY_Hv = fac * intX, fac * intY 
        out_units = "kg m^-4 s^-2" 
    # Optional: mask dry faces 
    wetU = (ds_grid["hFacW"] > 0).any("k") 
    wetV = (ds_grid["hFacS"] > 0).any("k") 
    dBuoyX_Hu = dBuoyX_Hu.where(wetU) 
    dBuoyY_Hv = dBuoyY_Hv.where(wetV) 
    dBuoyX_Hu.name, dBuoyY_Hv.name = "dBuoyX_Hu", "dBuoyY_Hv" 
    for da in (dBuoyX_Hu, dBuoyY_Hv): 
        da.attrs["long_name"] = "ML integral of horizontal buoyancy gradient" 
        da.attrs["units"] = out_units 
    return dBuoyX_Hu, dBuoyY_Hv

In [45]:
import numpy as np 
import xarray as xr 
def compute_Lf_MEI( dBuoyX_Hu: xr.DataArray, # (j, i_g) 
                    dBuoyY_Hv: xr.DataArray, # (j_g, i) 
                    b: xr.DataArray, # (k, j, i) buoyancy at C 
                    H_C: xr.DataArray, # (j, i) ML depth at C 
                    f_eff_C: xr.DataArray, # (j, i) 
                    ds_grid: xr.Dataset, 
                    grid, 
                    g: float = 9.81, 
                    rho0: float = 1035.0, 
                    LfMin: float = 1000.0, ): 
    # --- A) Standardize vertical coord 'k' everywhere ----------------------- 
    RF = ds_grid["RF"] # (k_p1) 
    K = RF.sizes["k_p1"] - 1 
    k_idx = xr.DataArray(np.arange(K), dims=("k",), name="k") 
    def to_k(da): 
        # rename vertical dim to 'k' if needed, then assign common index 
        for cand in ("k", "Z", "z", "lev", "k_c"): 
            if cand in da.dims: 
                if cand != "k": 
                    da = da.rename({cand: "k"}) 
                break 
        if "k" in da.dims: 
                da = da.assign_coords(k=k_idx).reset_coords(drop=True) 
        return da 
    b = to_k(b) 
    hFacC = to_k(ds_grid["hFacC"]) # (k,j,i) with unified k 
    # (dBuoyX_Hu, dBuoyY_Hv, H_C, f_eff_C are 2D—no k to fix) 
    # interfaces as positive-down depths with SAME 'k' coords 
    ztop1d = (-RF.isel(k_p1=slice(0, -1))).rename(k_p1="k").assign_coords(k=k_idx) 
    zbot1d = (-RF.isel(k_p1=slice(1, None))).rename(k_p1="k").assign_coords(k=k_idx) 
    ztop = ztop1d.broadcast_like(b) 
    zbot = zbot1d.broadcast_like(b) 
    # --- 1) MsqH at centers from face integrals averaged to C --------------- 
    dBx_C = grid.interp(dBuoyX_Hu, 'X') # U->C 
    dBy_C = grid.interp(dBuoyY_Hv, 'Y') # V->C 
    MsqH = np.sqrt(0.25*(dBx_C**2 + dBy_C**2)) # s^-2 m 
    # --- 2) NHmixLay at C via ML overlap (honor partial cells) -------------- 
    thkC = (zbot - ztop) * hFacC # partial thickness 
    H3C = H_C.assign_coords().broadcast_like(b) # (k,j,i) with unified k 
    # min(H,zbot) without xr.minimum (avoids version issues) 
    minHC = xr.where(H3C <= zbot, H3C, zbot) 
    wC = xr.where(H3C > ztop, minHC - ztop, 0.0) 
    wC = xr.where(wC > 0, xr.where(wC < thkC, wC, thkC), 0.0) 
    # ∂b/∂z at C using interface spacing (downward-positive dz) 
    db = b - b.shift(k=1) 
    dz = (ztop - ztop.shift(k=1)) 
    sigmaR = (db / dz).where(dz.notnull()) 
    sigmaR_pos = xr.where(sigmaR < 0.0, -sigmaR, 0.0) 
    NH_term = (sigmaR_pos * wC).sum("k") # units: s^-2 * m 
    NHmixLay = np.sqrt(NH_term * H_C.clip(min=0.0)).fillna(0.0) 
    # --- 3) Lf at centers --------------------------------------------------- 
    fC = xr.where(f_eff_C > 0, f_eff_C, np.nan) 
    cand1 = MsqH / (fC**2) # m 
    cand2 = NHmixLay / fC # m 
    Lf_center = xr.ufuncs.maximum( xr.ufuncs.maximum(cand1.fillna(0.0), cand2.fillna(0.0)), LfMin ).rename("Lf") 
    Lf_center.attrs.update(long_name="MEI length scale at centers", units="m") 
    # --- 4) centers -> faces for streamfunction denominators ---------------- 
    Lf_U = grid.interp(Lf_center, 'X', to='left') # C->U 
    Lf_V = grid.interp(Lf_center, 'Y', to='left') # C->V 
    Lf_U.name, Lf_V.name = "Lf_U", "Lf_V" 
    return Lf_center, Lf_U, Lf_V

# Calculate FK heat flux

In [42]:
data = np.load("/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/Eta_12snap_avg_coarse.npz")

# 看里面有哪些变量
print(data.files)

['U_coarse', 'block_start_time_index', 'block_end_time_index', 'block_size']


In [46]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("/scratch/jczhang/submeso_data_driven").resolve()))

import importlib
import numpy as np
import xarray as xr
import gsw

from MITgcmutils import mds
from src.utils import make_grid
from src.utils import build_coarse_grid
from src.utils import read_mitgcm
from src.utils import coarsen_UVWTS
from src.utils import mld_avg

importlib.reload(coarsen_UVWTS)

# =========================================================
# batch FK calculation from npz inputs
# =========================================================
def compute_fk_from_npz(
    npz_kpphbl,
    npz_T,
    npz_S,
    npz_Eta,
    out_npz,
    path_scratch="/scratch/jczhang/",
    path_MITgcm="MITgcm_1km/verification/tutorial_reentrant_channel/output",
    box_i=50,
    box_j=50,
    C_e=0.1485399,
    submeso_Lmax=1000.0,
    LfMin=110000.0,
    f0=-1.363e-4,
    beta=1.313e-11,
    subMeso_invTau=2.0e-6,
):
    """
    Inputs:
        npz_kpphbl: contains kpphbl_coarse (time,j,i), and time_index
        npz_T     : contains T_coarse      (time,k,j,i), and time_index
        npz_S     : contains S_coarse      (time,k,j,i), and time_index
        npz_Eta   : contains Eta_coarse    (time,j,i), and time_index

    Outputs:
        submeso_wT_FK_all : (time,j,i)
        Fz_b_ml_all       : (time,j,i)
        time_index        : (time,)
    """

    grid_dir = os.path.join(path_scratch, path_MITgcm)

    # --- load coarse grid ---
    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)
    ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine, box_j=box_j, box_i=box_i)
    grid = make_grid.make_grid_periodic_x(ds_coarse)

    # --- load npz ---
    d_kpp = np.load(npz_kpphbl)
    d_T = np.load(npz_T)
    d_S = np.load(npz_S)
    d_E = np.load(npz_Eta)

    kpphbl_all = d_kpp["kpphbl_coarse"]     # (time,j,i)
    T_all = d_T["U_coarse"]                 # (time,k,j,i)
    S_all = d_S["U_coarse"]                 # (time,k,j,i)
    Eta_all = d_E["U_coarse"]             # (time,j,i)

    time_kpp = d_kpp["time_index"]
    time_T = d_T["block_end_time_index"]
    time_S = d_S["block_end_time_index"]
    time_E = d_E["block_end_time_index"]

    # --- consistency check ---
    if not (np.array_equal(time_kpp, time_T) and np.array_equal(time_kpp, time_S) and np.array_equal(time_kpp, time_E)):
        raise ValueError("time_index in the 4 npz files are not identical.")

    time_index = time_kpp
    nt = len(time_index)

    # --- static Coriolis terms on coarse C-grid ---
    YC_da = xr.DataArray(ds_coarse["YC"], dims=("j", "i"))
    y_ref = float(YC_da.mean())
    f = f0 + beta * (YC_da - y_ref)
    f_eff = np.hypot(f, subMeso_invTau)

    # --- output lists ---
    submeso_wT_FK_list = []
    Fz_b_ml_list = []

    for it in range(nt):
        t = int(time_index[it])
        print(f"[{it+1}/{nt}] computing FK for t = {t}")

        # ----------------------------
        # build snapshot datasets
        # ----------------------------
        kpphbl_coarse = xr.DataArray(
            kpphbl_all[it],
            dims=("j", "i"),
            coords={"j": ds_coarse["YC"].coords["j"], "i": ds_coarse["XC"].coords["i"]},
            name="kpphbl",
        )

        T_coarse = xr.DataArray(
            T_all[it],
            dims=("k", "j", "i"),
            coords={"k": ds_coarse["RC"].coords["k"], "j": ds_coarse["YC"].coords["j"], "i": ds_coarse["XC"].coords["i"]},
            name="T",
        )

        S_coarse = xr.DataArray(
            S_all[it],
            dims=("k", "j", "i"),
            coords={"k": ds_coarse["RC"].coords["k"], "j": ds_coarse["YC"].coords["j"], "i": ds_coarse["XC"].coords["i"]},
            name="S",
        )

        Eta_coarse = xr.DataArray(
            Eta_all[it],
            dims=("j", "i"),
            coords={"j": ds_coarse["YC"].coords["j"], "i": ds_coarse["XC"].coords["i"]},
            name="Eta",
        )

        ds_ocean_c = xr.Dataset(dict(T=T_coarse, S=S_coarse, Eta=Eta_coarse))

        # ----------------------------
        # buoyancy + ML integrals
        # ----------------------------
        b, dBdx_u, dBdy_V = buo_coarse(ds_coarse, ds_ocean_c, "fill")

        dBuoyX_Hu, dBuoyY_Hv = mld_integrals_for_MEI(
            dBdx_u, dBdy_V, kpphbl_coarse, ds_coarse, grid, sigma_is_buoyancy=True
        )

        # ----------------------------
        # Coriolis on faces
        # ----------------------------
        f_U = grid.interp(f_eff, "X", to="left")
        f_V = grid.interp(f_eff, "Y", to="left")

        # ----------------------------
        # Lf
        # ----------------------------
        Lf_center, Lf_U, Lf_V = compute_Lf_MEI(
            dBuoyX_Hu=dBuoyX_Hu,
            dBuoyY_Hv=dBuoyY_Hv,
            b=b,
            H_C=kpphbl_coarse.clip(min=0.0),
            f_eff_C=f_eff,
            ds_grid=ds_coarse,
            grid=grid,
            LfMin=LfMin,
        )

        # ----------------------------
        # mu(z/H)
        # ----------------------------
        rF = ds_coarse["RF"].rename(k_p1="k")
        rF_U = rF.broadcast_like(dBdx_u)
        rF_V = rF.broadcast_like(dBdy_V)

        H_U = grid.interp(kpphbl_coarse.clip(min=0.0), "X", to="left")
        H_V = grid.interp(kpphbl_coarse.clip(min=0.0), "Y", to="left")

        z2H_U = xr.where(H_U > 0, 2.0 * rF_U / H_U.broadcast_like(dBdx_u), 0.0)
        z2H_V = xr.where(H_V > 0, 2.0 * rF_V / H_V.broadcast_like(dBdy_V), 0.0)

        def mu_from(z2H):
            q = (1.0 + z2H) ** 2
            mu = (1.0 - q) * (1.0 + (5.0 / 21.0) * q)
            return xr.where(mu > 0.0, mu, 0.0)

        muU = mu_from(z2H_U)
        muV = mu_from(z2H_V)

        # ----------------------------
        # streamfunctions
        # ----------------------------
        dx_cap_U = ds_coarse["DXC"].clip(max=submeso_Lmax)
        dy_cap_V = ds_coarse["DYC"].clip(max=submeso_Lmax)

        dBuoyY_on_U = grid.interp(dBuoyY_Hv, "Y")
        dBuoyY_on_U = grid.interp(dBuoyY_on_U, "X", to="left")

        dBuoyX_on_V = grid.interp(dBuoyX_Hu, "X")
        dBuoyX_on_V = grid.interp(dBuoyX_on_V, "Y", to="left")

        PsiU0 = (C_e * dBuoyY_on_U * H_U * dx_cap_U / (Lf_U * f_U)).fillna(0.0)
        PsiV0 = (-C_e * dBuoyX_on_V * H_V * dy_cap_V / (Lf_V * f_V)).fillna(0.0)

        muU_ = muU.transpose("k", "j", "i_g")
        muV_ = muV.transpose("k", "j_g", "i")

        PsiX = PsiU0.broadcast_like(muU_) * muU_
        PsiY = PsiV0.broadcast_like(muV_) * muV_

        # ----------------------------
        # temperature gradients
        # ----------------------------
        wetU = ds_coarse["hFacW"] > 0
        wetV = ds_coarse["hFacS"] > 0

        dTdx_U = (grid.diff(T_coarse, "X") / ds_coarse["DXC"]).where(wetU)
        dTdy_V = (grid.diff(T_coarse, "Y", boundary="fill") / ds_coarse["DYC"]).where(wetV)

        dTdx_C = grid.interp(dTdx_U.fillna(0.0), "X")
        dTdy_C = grid.interp(dTdy_V.fillna(0.0), "Y")

        dTdx_W = 0.5 * (dTdx_C + dTdx_C.shift(k=1))
        dTdy_W = 0.5 * (dTdy_C + dTdy_C.shift(k=1))

        PsiX_C = grid.interp(PsiX, "X")
        PsiY_C = grid.interp(PsiY, "Y")

        # ----------------------------
        # vertical heat flux
        # ----------------------------
        Fz_T_cross = (PsiX_C * dTdy_W - PsiY_C * dTdx_W).rename("Fz_T_cross")

        rho0, cp = 1035.0, 3990.0
        Qz_T_cross = (rho0 * cp) * Fz_T_cross
        Qz_T_cross.name = "Qz_T_cross"
        Qz_T_cross.attrs.update(units="W m^-2")

        submeso_wT_FK = mld_avg.mld_avg(Qz_T_cross, kpphbl_coarse, ds_coarse)

        # ----------------------------
        # vertical buoyancy flux
        # ----------------------------
        dbdx_C = grid.interp(dBdx_u.fillna(0.0), "X")
        dbdy_C = grid.interp(dBdy_V.fillna(0.0), "Y")

        dbdx_W = 0.5 * (dbdx_C + dbdx_C.shift(k=1))
        dbdy_W = 0.5 * (dbdy_C + dbdy_C.shift(k=1))

        Fz_b = (PsiX_C * dbdy_W - PsiY_C * dbdx_W).rename("Fz_b")
        Fz_b_ml = mld_avg.mld_avg(Fz_b, kpphbl_coarse, ds_coarse)

        # ----------------------------
        # append
        # ----------------------------
        submeso_wT_FK_list.append(np.asarray(submeso_wT_FK.values, dtype=np.float32))
        Fz_b_ml_list.append(np.asarray(Fz_b_ml.values, dtype=np.float32))

    # --- stack outputs ---
    submeso_wT_FK_all = np.stack(submeso_wT_FK_list, axis=0).astype(np.float32)
    Fz_b_ml_all = np.stack(Fz_b_ml_list, axis=0).astype(np.float32)

    # --- save ---
    np.savez_compressed(
        out_npz,
        submeso_wT_FK=submeso_wT_FK_all,
        Fz_b_ml=Fz_b_ml_all,
        time_index=time_index.astype(np.int64),
    )

    print(f"Saved FK outputs to: {out_npz}")
    print("submeso_wT_FK shape:", submeso_wT_FK_all.shape)
    print("Fz_b_ml shape:", Fz_b_ml_all.shape)

    return submeso_wT_FK_all, Fz_b_ml_all, time_index

In [47]:
submeso_wT_FK_all, Fz_b_ml_all, time_index = compute_fk_from_npz(
    npz_kpphbl="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/kpphbl_12snap_avg_coarse.npz",
    npz_T="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/T_12snap_avg_coarse.npz",
    npz_S="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/S_12snap_avg_coarse.npz",
    npz_Eta="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/Eta_12snap_avg_coarse.npz",
    out_npz="/scratch/jczhang/submeso_data_driven/src/post_analysis/05_Generalization_test/FK_heatflux.npz",
    box_i=50,
    box_j=50,
    C_e=0.1485399,
    submeso_Lmax=110000.0,
    LfMin=1000.0,
    subMeso_invTau=2.0e-6,
)

[1/72] computing FK for t = 1203120
[2/72] computing FK for t = 1204200
[3/72] computing FK for t = 1205280
[4/72] computing FK for t = 1206360
[5/72] computing FK for t = 1207440
[6/72] computing FK for t = 1208520
[7/72] computing FK for t = 1209600
[8/72] computing FK for t = 1210680
[9/72] computing FK for t = 1211760
[10/72] computing FK for t = 1212840
[11/72] computing FK for t = 1213920
[12/72] computing FK for t = 1215000
[13/72] computing FK for t = 1216080
[14/72] computing FK for t = 1217160
[15/72] computing FK for t = 1218240
[16/72] computing FK for t = 1219320
[17/72] computing FK for t = 1220400
[18/72] computing FK for t = 1221480
[19/72] computing FK for t = 1222560
[20/72] computing FK for t = 1223640
[21/72] computing FK for t = 1224720
[22/72] computing FK for t = 1225800
[23/72] computing FK for t = 1226880
[24/72] computing FK for t = 1227960
[25/72] computing FK for t = 1229040
[26/72] computing FK for t = 1230120
[27/72] computing FK for t = 1231200
[28/72] co

# use the other codes to save the inputs


In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("/scratch/jczhang/submeso_data_driven").resolve()))

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from src.models.UNet_ConvLSTM_4En4De_pixel import UNetPhysDepth1
from src.utils.machine_learning.evaluation_dist import eval_r2_suite


def load_unet_checkpoint(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    model = UNetPhysDepth1(
        input_channels=8,
        head_channel=1,
        base=256,
        kernel_size=3,
        activation="SiLU",
        batch_norm="bn",
        dropout_2D=0.2,
        use_lstm=False,
        lstm_dropout=0.15,
    ).to(device)

    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    norm = ckpt.get("norm", None)
    return model, norm


def evaluate_72_snapshots(
    ckpt_path,
    X,
    y,
    batch_size=16,
    denorm_y=False,
    save_npy="r2_each_snapshot.npy",
    save_csv="r2_each_snapshot.csv",
):
    """
    X: numpy array, shape (72, 8, 20, 20)
    y: numpy array, shape (72, 1, 20, 20)
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # ---------------- load model ----------------
    model, norm = load_unet_checkpoint(ckpt_path, device)

    # ---------------- check shapes ----------------
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    if X.shape != (72, 8, 20, 20):
        print(f"Warning: X shape is {X.shape}, not (72, 8, 20, 20)")
    if y.shape != (72, 1, 20, 20):
        print(f"Warning: y shape is {y.shape}, not (72, 1, 20, 20)")

    # ---------------- normalize X ----------------
    if norm is not None:
        if "X_mu" not in norm or "X_sd" not in norm:
            raise KeyError("norm must contain 'X_mu' and 'X_sd'")

        X_mu = np.asarray(norm["X_mu"], dtype=np.float32)
        X_sd = np.asarray(norm["X_sd"], dtype=np.float32)

        # expected shape usually (8,1,1) or (8,H,W) or broadcastable to X
        X = (X - X_mu) / (X_sd + 1e-12)

    # ---------------- make dataloader ----------------
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)

    dataset = TensorDataset(X_t, y_t)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    # ---------------- evaluate ----------------
    out = eval_r2_suite(
        model=model,
        loader=loader,
        device=device,
        norm=norm,
        denorm_y=denorm_y,
    )

    # ---------------- print summary ----------------
    print("\n===== Evaluation summary =====")
    print("pixel_r2       :", out["pixel_r2"])
    print("image_r2_mean  :", out["image_r2_mean"])
    print("image_r2_median:", out["image_r2_median"])
    print("climate_r2     :", out["climate_r2"])
    print("n_images       :", out["n_images"])
    print("n_pixels       :", out["n_pixels"])

    # ---------------- per-image R2 ----------------
    r2_each = np.asarray(out["image_r2_list"], dtype=np.float64)

    print("\n===== Per-snapshot R2 =====")
    for i, r2 in enumerate(r2_each):
        print(f"snapshot {i:02d}: {r2:.6f}")

    # ---------------- save ----------------
    np.save(save_npy, r2_each)
    np.savetxt(save_csv, r2_each, delimiter=",", header="r2", comments="")

    print(f"\nSaved per-image R2 to:")
    print("  ", save_npy)
    print("  ", save_csv)

    return out, r2_each


if __name__ == "__main__":
    ckpt_path = "/scratch/jczhang/submeso_data_driven/check_point/cfg_19/best_unet_conv.pt"

    X = np.load("X.npy")   # (72, 8, 20, 20)
    y = np.load("y.npy")   # (72, 1, 20, 20)

    out, r2_each = evaluate_72_snapshots(
        ckpt_path=ckpt_path,
        X=X,
        y=y,
        batch_size=16,
        denorm_y=False,
        save_npy="r2_each_snapshot.npy",
        save_csv="r2_each_snapshot.csv",
    )

Using device: cpu

===== Evaluation summary =====
pixel_r2       : -0.4116430604798007
image_r2_mean  : -0.43200466970860574
image_r2_median: -0.44624423836840976
climate_r2     : -0.8610151762520024
n_images       : 72
n_pixels       : 28800

===== Per-snapshot R2 =====
snapshot 00: -0.582995
snapshot 01: -0.562755
snapshot 02: -0.551640
snapshot 03: -0.547329
snapshot 04: -0.547473
snapshot 05: -0.546582
snapshot 06: -0.530798
snapshot 07: -0.516053
snapshot 08: -0.522566
snapshot 09: -0.509398
snapshot 10: -0.504396
snapshot 11: -0.477978
snapshot 12: -0.471491
snapshot 13: -0.511243
snapshot 14: -0.506153
snapshot 15: -0.463350
snapshot 16: -0.473313
snapshot 17: -0.462296
snapshot 18: -0.408730
snapshot 19: -0.418306
snapshot 20: -0.368759
snapshot 21: -0.340515
snapshot 22: -0.357441
snapshot 23: -0.392611
snapshot 24: -0.442057
snapshot 25: -0.457190
snapshot 26: -0.442354
snapshot 27: -0.455186
snapshot 28: -0.401561
snapshot 29: -0.340426
snapshot 30: -0.323043
snapshot 31: -0